### Collecting ownership history of franchinses throughout time from RealGM and Basketball Reference

In [2]:
import pandas as pd
import time


owners = pd.read_csv("../data/raw/Ownership/NBA Ownership Tenures .csv")

len(owners)

print(owners.shape)
owners.head()
multiple_team_mask = owners["Team"].str.contains(",", na=False)
multi_team_owners = owners.loc[multiple_team_mask]

multi_team_owners

(156, 7)


,Staff,Team,Start Season,End Season,Years In Role,Birth Date,Nationality
15,John Brown,"LAC, BOS",1976-1977,1979-1980,4,-,-
25,Alan Cohen,"BRK, BOS",1978-1979,1992-1993,15,-,-
49,Jim Fitzgerald,"MIL, GOS",1975-1976,1993-1994,18,-,-
88,Irv Levin,"BOS, LAC",1974-1975,1980-1981,7,-,-
94,"Harry Mangurian, Jr.","LAC, BOS",1977-1978,1982-1983,6,-,-
96,Red McCombs,"DEN, SAN",1978-1979,1992-1993,12,-,-
101,Gail Miller,"UTH, UTH",2009-2010,2020-2021,11,-,United States
149,Jack Waldron,"BOS, BOS, BOS",1965-1966,1971-1972,4,-,-


In [4]:
owners_list = owners.copy()

col_names = ["owner_name", "team_id", "start_season", "end_season", "tenure", "birth_date", "nationality"]

owners_list.columns = col_names

owners_list[["owner_name", "team_id", "start_season", "end_season", "tenure"]]

owners_list.head()
owners_list.info()

<class 'pandas.DataFrame'>
RangeIndex: 156 entries, 0 to 155
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   owner_name    156 non-null    str  
 1   team_id       156 non-null    str  
 2   start_season  156 non-null    str  
 3   end_season    156 non-null    str  
 4   tenure        156 non-null    int64
 5   birth_date    156 non-null    str  
 6   nationality   156 non-null    str  
dtypes: int64(1), str(6)
memory usage: 8.7 KB


In [10]:
# Extract the starting year of each NBA season.
owners_list["start_year"] = pd.to_numeric(
    owners_list["start_season"].str[:4],
    errors="coerce"
)

# Extract the starting year of the final NBA season.
owners_list["end_year"] = pd.to_numeric(
    owners_list["end_season"].str[:4],
    errors="coerce"
)

# Started on or before the last season in our study.
started_in_time = owners_list["start_year"] <= 2025

# Ended during or after the first season in our study.
ended_in_time = owners_list["end_year"] >= 2001

# Source explicitly uses "-" for ongoing ownership.
still_active = owners_list["end_season"].astype("string").str.strip().eq("-")

# Keep owners whose tenure overlaps our research window.
overlap_mask = started_in_time & (ended_in_time | still_active)

# Apply the filter to the original cleaned DataFrame.
owners_filtered = owners_list.loc[overlap_mask].copy()

In [11]:
display(owners_filtered)

,owner_name,team_id,start_season,end_season,tenure,birth_date,nationality,start_year,end_year
1,Leslie Alexander,HOU,1993-1994,2016-2017,24,-,United States,1993,2016.0
2,Paul Allen,POR,1988-1989,2017-2018,30,"Jan 21, 1953",United States,1988,2017.0
3,Steve Ballmer,LAC,2014-2015,-,13,"Mar 24, 1956",United States,2014,NaN
5,Clay Bennett,OKC,2006-2007,-,21,-,-,2006,NaN
6,Tom Benson,NOP,2011-2012,2017-2018,7,"Jul 12, 1927",United States,2011,2017.0
...,...,...,...,...,...,...,...,...,...
141,Glen Taylor,MIN,1995-1996,2024-2025,30,-,United States,1995,2024.0
144,Legacy Trust,UTH,2016-2017,2017-2018,2,-,United States,2016,2017.0
145,Paul G. Allen Trust,POR,2018-2019,2025-2026,8,-,United States,2018,2025.0
146,Joseph Tsai,BRK,2017-2018,-,10,-,United States,2017,NaN


In [12]:
display(
    owners_list.loc[
        owners_list["end_year"].isna(),
        ["owner_name", "team_id", "end_season"]
    ]
)

display(len(owners_list["end_year"].isna()))

,owner_name,team_id,end_season
3,Steve Ballmer,LAC,-
5,Clay Bennett,OKC,-
10,Sara Blakely,ATL,-
37,James L. Dolan,NYK,-
39,Clayton Dubilier,ATL,-
45,Tilman Fertitta,HOU,-
52,Dan Gilbert,CLE,-
56,Tom Gores,DET,-
59,Wyc Grousbeck,BOS,-
60,Peter Guber,GOS,-


156

In [13]:
print("Original rows:", len(owners_list))
print("Filtered rows:", len(owners_filtered))

display(owners_filtered.head())
display(owners_filtered.tail())

Original rows: 156
Filtered rows: 67


,owner_name,team_id,start_season,end_season,tenure,birth_date,nationality,start_year,end_year
1,Leslie Alexander,HOU,1993-1994,2016-2017,24,-,United States,1993,2016.0
2,Paul Allen,POR,1988-1989,2017-2018,30,"Jan 21, 1953",United States,1988,2017.0
3,Steve Ballmer,LAC,2014-2015,-,13,"Mar 24, 1956",United States,2014,NaN
5,Clay Bennett,OKC,2006-2007,-,21,-,-,2006,NaN
6,Tom Benson,NOP,2011-2012,2017-2018,7,"Jul 12, 1927",United States,2011,2017.0


,owner_name,team_id,start_season,end_season,tenure,birth_date,nationality,start_year,end_year
141,Glen Taylor,MIN,1995-1996,2024-2025,30,-,United States,1995,2024.0
144,Legacy Trust,UTH,2016-2017,2017-2018,2,-,United States,2016,2017.0
145,Paul G. Allen Trust,POR,2018-2019,2025-2026,8,-,United States,2018,2025.0
146,Joseph Tsai,BRK,2017-2018,-,10,-,United States,2017,NaN
147,Ted Turner,ATL,1976-1977,2004-2005,29,-,-,1976,2004.0


In [16]:
print("Original rows:", len(owners_list))
print("Previously filtered rows:", 43)
print("Updated filtered rows:", len(owners_filtered))

print(
    "Active owners retained:",
    owners_filtered["end_season"].astype("string").str.strip().eq("-").sum()
)

print(
    "Unique team IDs:",
    owners_filtered["team_id"].nunique()
)

display(owners_filtered.tail(15))

Original rows: 156
Previously filtered rows: 43
Updated filtered rows: 67
Active owners retained: 24
Unique team IDs: 29


,owner_name,team_id,start_season,end_season,tenure,birth_date,nationality,start_year,end_year
119,Robert Sarver,PHX,2004-2005,2022-2023,19,-,United States,2004,2022.0
122,Rick Schnall,ATL,2015-2016,2022-2023,8,-,United States,2015,2022.0
125,Howard Schultz,OKC,2001-2002,2005-2006,5,-,-,2001,2005.0
126,George Shinn,NOP,1988-1989,2010-2011,23,-,-,1988,2010.0
129,Herbert Simon,IND,1983-1984,-,44,"Oct 20, 1934",United States,1983,NaN
130,Mel Simon,IND,1983-1984,2008-2009,26,-,-,1983,2008.0
131,Stephen Simon,IND,2017-2018,-,10,-,United States,2017,NaN
133,Ryan Smith,UTH,2020-2021,-,7,-,United States,2020,NaN
134,Ed Snider,PHL,1996-1997,2010-2011,15,-,-,1996,2010.0
137,Donald Sterling,LAC,1981-1982,2013-2014,33,-,-,1981,2013.0


In [17]:
owners_filtered.loc[
    (owners_filtered["owner_name"] == "Gail Miller") &
    (owners_filtered["team_id"] == "UTH, UTH"),
    "team_id"
] = "UTH"

In [18]:
owners_filtered["team_id"].nunique()

28

In [19]:
display(
    owners_filtered.loc[
        owners_filtered["team_id"].str.contains(",", na=False)
    ]
)

,owner_name,team_id,start_season,end_season,tenure,birth_date,nationality,start_year,end_year


In [20]:
# Show each unique team identifier in alphabetical order.
print(sorted(owners_filtered["team_id"].dropna().unique()))

['ATL', 'BOS', 'BRK', 'CHA', 'CHI', 'CLE', 'DEN', 'DET', 'GOS', 'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIL', 'MIN', 'NOP', 'NYK', 'OKC', 'ORL', 'PHL', 'PHX', 'POR', 'SAC', 'SAN', 'TOR', 'UTH', 'WAS']


In [21]:
# Count ownership records per team identifier.
team_counts = (
    owners_filtered["team_id"]
    .value_counts()
    .sort_index()
)

display(team_counts)

team_id
ATL    7
BOS    5
BRK    5
CHA    2
CHI    1
CLE    3
DEN    1
DET    3
GOS    3
HOU    2
IND    3
LAC    2
LAL    2
MEM    2
MIL    1
MIN    3
NOP    2
NYK    1
OKC    2
ORL    1
PHL    1
PHX    3
POR    2
SAC    2
SAN    2
TOR    1
UTH    4
WAS    1
Name: count, dtype: int64

In [22]:
# Expected franchise IDs from our S15 project.
expected_teams = {
    "ATL", "BOS", "BRK", "CHI", "CHO", "CLE",
    "DAL", "DEN", "DET", "GSW", "HOU", "IND",
    "LAC", "LAL", "MEM", "MIA", "MIL", "MIN",
    "NOP", "NYK", "OKC", "ORL", "PHI", "PHO",
    "POR", "SAC", "SAS", "TOR", "UTA", "WAS"
}

# Extract the observed RealGM identifiers.
observed_teams = set(owners_filtered["team_id"].dropna())

# Identify codes that don't match our canonical convention.
unexpected_codes = observed_teams - expected_teams

# Identify canonical codes that aren't present.
missing_codes = expected_teams - observed_teams

print("Unexpected RealGM codes:", sorted(unexpected_codes))
print("Missing canonical codes:", sorted(missing_codes))

Unexpected RealGM codes: ['CHA', 'GOS', 'PHL', 'PHX', 'SAN', 'UTH']
Missing canonical codes: ['CHO', 'DAL', 'GSW', 'MIA', 'PHI', 'PHO', 'SAS', 'UTA']


In [23]:
# Map RealGM team identifiers to our canonical franchise IDs.
team_code_map = {
    "CHA": "CHO",
    "GOS": "GSW",
    "PHL": "PHI",
    "PHX": "PHO",
    "SAN": "SAS",
    "UTH": "UTA"
}

# Standardize franchise identifiers.
owners_filtered["team_id"] = owners_filtered["team_id"].replace(team_code_map)

In [24]:
# Search the unfiltered source data for Dallas and Miami.
display(
    owners_list.loc[
        owners_list["team_id"].isin(["DAL", "MIA"])
    ]
)

,owner_name,team_id,start_season,end_season,tenure,birth_date,nationality,start_year,end_year
21,Don Carter,DAL,1980-1981,1995-1996,16,-,-,1980,1995.0
29,Billy Cunningham,MIA,1987-1988,1994-1995,8,"Jun 3, 1943",United States,1987,1994.0
107,"Ross Perot, Jr.",DAL,1996-1997,1998-1999,3,-,-,1996,1998.0
120,Lewis Schaffel,MIA,1987-1988,1994-1995,8,-,-,1987,1994.0


In [25]:
print(sorted(owners_list["team_id"].dropna().unique()))

['ATL', 'BOS', 'BOS, BOS, BOS', 'BOS, LAC', 'BRK', 'BRK, BOS', 'CHA', 'CHI', 'CLE', 'DAL', 'DEN', 'DEN, SAN', 'DET', 'GOS', 'HOU', 'IND', 'LAC', 'LAC, BOS', 'LAL', 'MEM', 'MIA', 'MIL', 'MIL, GOS', 'MIN', 'NOP', 'NYK', 'OKC', 'ORL', 'PHL', 'PHX', 'POR', 'SAC', 'SAN', 'TOR', 'UTH', 'UTH, UTH', 'WAS']


In [26]:
# Compare standardized ownership codes with the expected 30 franchises.
observed_teams = set(owners_filtered["team_id"].dropna())

print("Unique franchises:", len(observed_teams))
print("Missing franchises:", sorted(expected_teams - observed_teams))
print("Unexpected codes:", sorted(observed_teams - expected_teams))
print("Ownership records:", len(owners_filtered))

Unique franchises: 28
Missing franchises: ['DAL', 'MIA']
Unexpected codes: []
Ownership records: 67


In [27]:
# Manually verified supplemental ownership records.
supplemental_owners = pd.DataFrame([
    {
        "owner_name": "Mark Cuban",
        "team_id": "DAL",
        "start_season": "2000-2001",
        "end_season": "2023-2024"
    },
    {
        "owner_name": "Adelson-Dumont family",
        "team_id": "DAL",
        "start_season": "2023-2024",
        "end_season": "-"
    },
    {
        "owner_name": "Micky Arison",
        "team_id": "MIA",
        "start_season": "1995-1996",
        "end_season": "-"
    }
])

# Derive numeric years using the same rules as the original data.
supplemental_owners["start_year"] = pd.to_numeric(
    supplemental_owners["start_season"].str[:4]
)

supplemental_owners["end_year"] = pd.to_numeric(
    supplemental_owners["end_season"].str[:4],
    errors="coerce"
)

# Add a source flag to distinguish original and supplemental records.
owners_filtered["source"] = "RealGM ownership CSV"
supplemental_owners["source"] = "Supplemental verified history"

# Combine the datasets.
owners_final = pd.concat(
    [owners_filtered, supplemental_owners],
    ignore_index=True
)

In [28]:
print("Total ownership records:", len(owners_final))
print("Unique franchises:", owners_final["team_id"].nunique())
print("Missing franchises:", sorted(
    expected_teams - set(owners_final["team_id"])
))

display(
    owners_final.loc[
        owners_final["team_id"].isin(["DAL", "MIA"])
    ]
)

Total ownership records: 70
Unique franchises: 30
Missing franchises: []


,owner_name,team_id,start_season,end_season,tenure,birth_date,nationality,start_year,end_year,source
67,Mark Cuban,DAL,2000-2001,2023-2024,NaN,NaN,NaN,2000,2023.0,Supplemental verified history
68,Adelson-Dumont family,DAL,2023-2024,-,NaN,NaN,NaN,2023,NaN,Supplemental verified history
69,Micky Arison,MIA,1995-1996,-,NaN,NaN,NaN,1995,NaN,Supplemental verified history


In [29]:
# Keep the columns needed for ownership analysis.
owners_final = owners_final[
    [
        "owner_name",
        "team_id",
        "start_season",
        "end_season",
        "start_year",
        "end_year",
        "source"
    ]
].copy()

# Store season years as nullable integers.
# Int64 allows missing end years for ongoing owners.
owners_final["start_year"] = owners_final["start_year"].astype("Int64")
owners_final["end_year"] = owners_final["end_year"].astype("Int64")

# Final validation.
print("Shape:", owners_final.shape)
print("Unique franchises:", owners_final["team_id"].nunique())
print("Missing owner names:", owners_final["owner_name"].isna().sum())
print("Missing start years:", owners_final["start_year"].isna().sum())
print("Duplicate rows:", owners_final.duplicated().sum())

# Missing end years should correspond to explicitly ongoing tenures.
invalid_end_dates = (
    owners_final["end_year"].isna()
    & owners_final["end_season"].ne("-")
)

print("Unexplained missing end years:", invalid_end_dates.sum())

# Check for impossible season ordering.
invalid_tenures = (
    owners_final["end_year"].notna()
    & (owners_final["end_year"] < owners_final["start_year"])
)

print("Invalid tenure ordering:", invalid_tenures.sum())
print("Missing franchises:", sorted(
    expected_teams - set(owners_final["team_id"])
))

Shape: (70, 7)
Unique franchises: 30
Missing owner names: 0
Missing start years: 0
Duplicate rows: 0
Unexplained missing end years: 0
Invalid tenure ordering: 0
Missing franchises: []


In [30]:
from pathlib import Path

# Create an ownership subfolder within staging.
staging_dir = Path("../data/staging/ownership")
staging_dir.mkdir(parents=True, exist_ok=True)

# Export the cleaned ownership table.
output_path = staging_dir / "S16_Ownership_Tenures.csv"

owners_final.to_csv(output_path, index=False)

# Read it back to verify the export.
sanity_check = pd.read_csv(output_path)

print("Saved:", output_path)
print("Reloaded shape:", sanity_check.shape)
display(sanity_check.head())

Saved: ../data/staging/ownership/S16_Ownership_Tenures.csv
Reloaded shape: (70, 7)


,owner_name,team_id,start_season,end_season,start_year,end_year,source
0,Leslie Alexander,HOU,1993-1994,2016-2017,1993,2016.0,RealGM ownership CSV
1,Paul Allen,POR,1988-1989,2017-2018,1988,2017.0,RealGM ownership CSV
2,Steve Ballmer,LAC,2014-2015,-,2014,NaN,RealGM ownership CSV
3,Clay Bennett,OKC,2006-2007,-,2006,NaN,RealGM ownership CSV
4,Tom Benson,NOP,2011-2012,2017-2018,2011,2017.0,RealGM ownership CSV
